# Сравнение и выбор связки моделей

В данном ноутбуке проведен выбор оптимальной связки для модели E5(multilingual-e5-small). Выбор размера лучшего размера чанков не производился, тк уже был определен в первом эксперименте и существенных изменений не дал. Выбран размер 40/20.

Сравнивались меры сходства

- cosine
- dot
- Euclidean

Итоговая таблица сравнения по мерам сходства

| Конфигурация | MRR | Precision | Recall | MAP |
|-------------|-----|-----------|--------|-----|
| **E5_cosine** | **0.7126** | **0.3013** | **0.4922** | **0.3740** |
| E5_dot | 0.7126 | 0.3013 | 0.4922 | 0.3740 |
| E5_euclidean | 0.7126 | 0.3013 | 0.4922 | 0.3740 |

---
**Ключевые выводы**

Все метрики дали одинаковые результаты, потому что e5 обучена на dot и всегда нормализует входящие векторы. Из-за этого не было смысла смотреть на разные меры сходства для этой модели, так как каждая мера дает идентичные результаты.

В остальном модель показывает стабильные результаты, справляется почти так же, как и остальные.

---
**Выявленные проблемы**

Проблема с плохой обработкой запросов с опечатками остается.

Модель путает тематики обращений.


# Вспомогательные функции

In [3]:
import numpy as np
import pandas as pd
import numpy as np
import json
import pandas as pd
import faiss
from typing import List, Optional, Tuple
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from IPython.display import display, Markdown
from IPython.display import display, Markdown


c:\Users\ivasi\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def chunk_text(text: str, chunk_size: int = 20, overlap: int = 5) -> List[str]:
    words = text.replace("\n", " ").split()

    if chunk_size <= 0:
        raise ValueError("chunk_size должен быть положительным.")
    if overlap >= chunk_size:
        raise ValueError("overlap должен быть меньше chunk_size.")

    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(words), step):
        chunk_words = words[start : start + chunk_size]
        if not chunk_words:
            continue

        chunks.append(" ".join(chunk_words))

        if start + chunk_size >= len(words):
            break

    return chunks

def buid_chunks(
    df_docs: pd.DataFrame,
    chunk_size: Optional[int] = None,
    overlap: int = 5
) -> pd.DataFrame:

    rows = []
    
    for _, row in df_docs.iterrows():
        doc_id = row['doc_id']
        topic = row['topic']
        text = row['response_text']
        
        if chunk_size is None:
            rows.append({
                'chunk_id': doc_id,
                'doc_id': doc_id,
                'topic': topic,
                'text': text
            })
        else:
            chunks = chunk_text(text, chunk_size=chunk_size, overlap=overlap)
            for i, chunk in enumerate(chunks):
                rows.append({
                    'chunk_id': f"{doc_id}_chunk_{i}",
                    'doc_id': doc_id,
                    'topic': topic,
                    'text': chunk
                })
    
    return pd.DataFrame(rows)


def get_metrics(
    retrieved_docs: List[str],
    relevant_docs: List[str],
    metrics: List[str] = ['precision', 'recall', 'mrr', 'map']
) -> pd.DataFrame:

    relevant_set = set(relevant_docs)
    total_relevant = len(relevant_set)
    
    hits = sum(1 for doc in retrieved_docs if doc in relevant_set)
    
    result = {}
    
    if 'precision' in metrics:
        result['precision'] = hits / len(retrieved_docs) if len(retrieved_docs) > 0 else np.nan
    
    if 'recall' in metrics:
        result['recall'] = hits / total_relevant if total_relevant > 0 else np.nan
    
    if 'mrr' in metrics:
        first_relevant_rank = None
        for idx, doc_id in enumerate(retrieved_docs, start=1):
            if doc_id in relevant_docs:
                first_relevant_rank = idx
                break
        result['mrr'] = 0.0 if first_relevant_rank is None else 1.0 / first_relevant_rank
    
    if 'map' in metrics:
        precisions = []
        hits_so_far = 0
        for i, doc in enumerate(retrieved_docs, 1):
            if doc in relevant_set:
                hits_so_far += 1
                precisions.append(hits_so_far / i)
        result['map'] = sum(precisions) / total_relevant if precisions and total_relevant > 0 else 0.0
    
    return pd.DataFrame([result])

In [5]:
class EmbeddingBackend:
    def __init__(self, model_name: str, device: str = "cpu", normalize: bool = True):
        self.model = SentenceTransformer(model_name, device=device)
        self.model_name = model_name
        self.normalize = normalize
    
    def encode(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=self.normalize
        )
        return vectors.astype("float32")

def build_embedding_backend(
    model_name: str = "paraphrase-multilingual-MiniLM-L12-v2",
    device: str = "cpu",
    normalize: bool = True
) -> EmbeddingBackend:
    try:
        backend = EmbeddingBackend(model_name=model_name, device=device, normalize=normalize)
        print(f"Модель: {model_name}, нормировка={normalize}")
        return backend
    except Exception as e:
        print(f"Ошибка загрузки {model_name}: {e}")
        raise

In [6]:
class VectorSearchIndex:
    def __init__(self, dim: int, similarity: str = "cosine"):

        self.dim = dim
        self.similarity = similarity
        self._faiss_index = None
        
        if similarity == "cosine":
            self._faiss_index = faiss.IndexFlatIP(dim)
        elif similarity == "euclidean":
            self._faiss_index = faiss.IndexFlatL2(dim)
        elif similarity == "dot":
            self._faiss_index = faiss.IndexFlatIP(dim)
    
    def add(self, vectors: np.ndarray) -> None:
        vectors = vectors.astype("float32")
        
        if self.similarity == "cosine":
            faiss.normalize_L2(vectors)
        
        self._faiss_index.add(vectors)
    
    def search(self, query_vectors: np.ndarray, top_k: int = 5) -> Tuple[np.ndarray, np.ndarray]:
        query_vectors = query_vectors.astype("float32")
        
        if self.similarity == "cosine":
            faiss.normalize_L2(query_vectors)
        
        scores, indices = self._faiss_index.search(query_vectors, top_k)
        
        if self.similarity == "euclidean":
            scores = -scores
        
        return scores, indices
    
def sanity_check_index(
    index: VectorSearchIndex,
    vectors: np.ndarray
):
    if index.similarity == "cosine":
        norms = np.linalg.norm(vectors, axis=1)
        print(f"Нормы документов: min={norms.min():.4f}, max={norms.max():.4f}")
        assert np.allclose(norms, 1.0, atol=1e-5), "Векторы не нормированы для cosine"

    print(f"{index.similarity}: векторы готовы для меры сходства, FAISS отработает корректно")

In [7]:
def evaluate_retrieval_bundle_pipeline(
    df_docs: pd.DataFrame,
    df_queries: pd.DataFrame,
    model_name: str,
    similarity: str,
    chunk_size: 40,
    overlap: int = 20,
    k: int = 5,
    device: str = "cpu"
) -> pd.DataFrame:

    print(f"Связка: модель={model_name.split('/')[-1]}, мера={similarity}, чанки={chunk_size if chunk_size else 'нет'}")

    
    df_chunks = buid_chunks(df_docs, chunk_size=chunk_size, overlap=overlap)
    print(f"Документов/чанков: {len(df_chunks)}")

    need_normalize = (similarity == "cosine")
    backend = build_embedding_backend(model_name, device=device, normalize=need_normalize)

    chunk_texts = df_chunks['text'].tolist()
    chunk_embeddings = backend.encode(chunk_texts)
    
    dim = chunk_embeddings.shape[1]
    index = VectorSearchIndex(dim, similarity=similarity)
    index.add(chunk_embeddings)
    
    sanity_check_index(index, chunk_embeddings)
    
    results = []
    
    for _, row in df_queries.iterrows():
        query = row['query_text']
        relevant_docs = row['relevant_docs'].split('|')
        
        query_vec = backend.encode([query])
        
        scores, indices = index.search(query_vec, top_k=k)
        
        predicted_chunk_ids = [df_chunks.iloc[idx]['chunk_id'] for idx in indices[0]]
        predicted_doc_ids = [cid.split('_chunk')[0] for cid in predicted_chunk_ids]
        
        metrics_df = get_metrics(
            retrieved_docs=predicted_doc_ids,
            relevant_docs=relevant_docs,
            metrics=['precision', 'recall', 'mrr', 'map']
        )
        
        results.append({
            'query_id': row['q_id'],
            'query': query,
            'relevant_docs': '|'.join(relevant_docs),
            'predicted_docs': '|'.join(predicted_doc_ids),
            'scores': '|'.join([f"{s:.4f}" for s in scores[0]]),
            'precision': metrics_df.iloc[0]['precision'],
            'recall': metrics_df.iloc[0]['recall'],
            'mrr': metrics_df.iloc[0]['mrr'],
            'map': metrics_df.iloc[0]['map']
        })
    
    return pd.DataFrame(results)



# Мера сходства

In [8]:
import logging
logging.getLogger("sentence_transformers").setLevel(logging.WARNING)
df_docs = pd.DataFrame(pd.read_csv('../data/documents.csv'))
df_queries = pd.DataFrame(pd.read_csv('../data/queries.csv'))
doc_text_map = dict(zip(df_docs['doc_id'], df_docs['response_text']))

e5_configs = [
    {'name': 'E5_cosine', 'similarity': 'cosine', 'chunk_size': 40, 'overlap': 20},
    {'name': 'E5_dot', 'similarity': 'dot', 'chunk_size': 40, 'overlap': 20},
    {'name': 'E5_euclidean', 'similarity': 'euclidean', 'chunk_size': 40, 'overlap': 20},
]
e5_results = []
best_mrr = -1
best_config = None
best_df = None

for cfg in e5_configs:
    display(Markdown(f"## {cfg['name']} | мера={cfg['similarity']}"))
        
    df_res = evaluate_retrieval_bundle_pipeline(
        df_docs=df_docs,
        df_queries=df_queries,
        model_name='intfloat/multilingual-e5-small',
        similarity=cfg['similarity'],
        chunk_size=cfg['chunk_size'],
        k=5,
        device='cpu'
    )
    
    df_res['first_predicted_id'] = df_res['predicted_docs'].apply(lambda x: x.split('|')[0] if pd.notna(x) else '')
    df_res['first_predicted'] = df_res['first_predicted_id'].map(doc_text_map).fillna('')
    df_res['first_relevant_id'] = df_res['relevant_docs'].apply(lambda x: x.split('|')[0] if pd.notna(x) else '')
    df_res['first_hit'] = df_res['first_predicted_id'] == df_res['first_relevant_id']

    display(Markdown(f"### Результаты для {cfg['name']}"))
    display_cols = ['query', 'relevant_docs', 'predicted_docs', 'scores', 'first_predicted', 'first_hit', 'mrr', 'precision', 'recall', 'map']
    
    display(Markdown(f"### Топ 3 лучших результатов (по MRR)"))
    top3 = df_res.nlargest(3, 'mrr')
    display(top3[display_cols])

    display(Markdown(f"### Топ 3 худших результатов (по MRR)"))

    worst3 = df_res.nsmallest(3, 'mrr')
    display(worst3[display_cols])

    avg_precision = df_res['precision'].mean()
    avg_recall = df_res['recall'].mean()
    avg_mrr = df_res['mrr'].mean()
    avg_map = df_res['map'].mean()
    
    e5_results.append({
        'config': cfg['name'],
        'similarity': cfg['similarity'],
        'chunk_size': cfg['chunk_size'] if cfg['chunk_size'] else 'full',
        'precision': avg_precision,
        'recall': avg_recall,
        'mrr': avg_mrr,
        'map': avg_map,
    })

    

## E5_cosine | мера=cosine

Связка: модель=multilingual-e5-small, мера=cosine, чанки=40
Документов/чанков: 40


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9045.72it/s]
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель: intfloat/multilingual-e5-small, нормировка=True
Нормы документов: min=1.0000, max=1.0000
cosine: векторы готовы для меры сходства, FAISS отработает корректно


### Результаты для E5_cosine

...

### Результаты для E5_euclidean

### Топ 3 лучших результатов (по MRR)

,query,relevant_docs,predicted_docs,scores,first_predicted,first_hit,mrr,precision,recall,map
2,"Не получается зайти в аккаунт, пароль не подхо...",doc_1|doc_5|doc_8,doc_5|doc_6|doc_4|doc_2|doc_8,-0.2393|-0.2455|-0.2477|-0.2583|-0.2606,Добрый день! Благодарим за вопрос. Если вы не ...,False,1.0,0.4,0.666667,0.466667
3,Привет! забыл пароль что делать подскажите,doc_1|doc_3|doc_6,doc_6|doc_5|doc_3|doc_8|doc_2,-0.2156|-0.2157|-0.2307|-0.2310|-0.2312,"Здравствуйте! Спасибо, что выбрали нашу платфо...",False,1.0,0.4,0.666667,0.555556
4,"Добрый день. Сменил пароль, теперь вообще не з...",doc_2|doc_5|doc_7,doc_5|doc_3|doc_6|doc_37|doc_4,-0.1726|-0.1794|-0.1965|-0.1965|-0.2061,Добрый день! Благодарим за вопрос. Если вы не ...,False,1.0,0.2,0.333333,0.333333


### Топ 3 худших результатов (по MRR)

,query,relevant_docs,predicted_docs,scores,first_predicted,first_hit,mrr,precision,recall,map
9,немогу зайти потму что пишет ошибка доступа,doc_1|doc_4|doc_5,doc_37|doc_40|doc_36|doc_38|doc_6,-0.2442|-0.2481|-0.2523|-0.2576|-0.2583,Добрый день! Спасибо за ваш вопрос. Если тест ...,False,0.0,0.0,0.0,0.0
66,Как получить чек об оплате за прошлый месяц? В...,doc_12|doc_14|doc_16,doc_13|doc_3|doc_6|doc_9|doc_29,-0.2559|-0.2647|-0.2746|-0.2770|-0.2810,Добрый день! Спасибо за ваш вопрос. Ошибка при...,False,0.0,0.0,0.0,0.0
81,При попытке загрузить файл пишет 'превышен мак...,doc_35|doc_36|doc_38,doc_25|doc_29|doc_34|doc_37|doc_40,-0.2157|-0.2177|-0.2354|-0.2601|-0.2626,Здравствуйте! Спасибо за обращение. Загрузка д...,False,0.0,0.0,0.0,0.0


In [9]:
summary_measure_df = pd.DataFrame(e5_results)

summary_cols = ['config', 'similarity', 'chunk_size', 'precision', 'recall', 'mrr', 'map']
summary_measure_df = summary_measure_df[summary_cols]

summary_measure_df = summary_measure_df.sort_values('mrr', ascending=False)

for col in ['precision', 'recall', 'mrr', 'map']:
    summary_measure_df[col] = summary_measure_df[col].round(4)

display(summary_measure_df)

best_row = summary_measure_df.iloc[0]

display(Markdown(f"### Лучшая конфигурация: {best_row['config']}"))
print(f"   MRR: {best_row['mrr']:.4f}")
print(f"   Precision: {best_row['precision']:.4f}")
print(f"   Recall: {best_row['recall']:.4f}")
print(f"   MAP: {best_row['map']:.4f}")

,config,similarity,chunk_size,precision,recall,mrr,map
0,E5_cosine,cosine,40,0.3013,0.4922,0.7126,0.374
1,E5_dot,dot,40,0.3013,0.4922,0.7126,0.374
2,E5_euclidean,euclidean,40,0.3013,0.4922,0.7126,0.374


### Лучшая конфигурация: E5_cosine

   MRR: 0.7126
   Precision: 0.3013
   Recall: 0.4922
   MAP: 0.3740


In [10]:
summary_measure_df.to_csv(
    "../artifacts/e5_summary_measure_df.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Файл сохранён: ./artifacts/e5_summary_measure_df.csv")

Файл сохранён: ./artifacts/e5_summary_measure_df.csv
